This notebook constructs the final feature sets (A, B, C, D, E1, E2) from `05_absa_bert_scores.csv`.

| Set | Core Features | NaN in Core |
|---|---|---|
| A | Numerical sub-ratings | Yes → skipped or not observed |
| B | VADER document-level scores | None |
| C | Rule-based aspect VADER scores | Yes → aspect not mentioned or insufficient tokens to compute a score |
| D | B + C | Yes → same as C |
| E1 | ABSA-BERT full inference scores | None |
| E2 | ABSA-BERT keyword-gated scores | Yes → aspect not mentioned or insufficient tokens to compute a score |


**How NaN is Handled**

- **Set A:** In `01_eda_cleaning.ipynb`, all 0 values in numerical columns were replaced with `None`, since the rating scale is 1–5 and 0 is not a valid observed value. These NaNs will be imputed in the modelling pipeline based on train dataset to prevent data leakage.

- **Set C, D, E2:** No imputation since there is no valid statistical basis for borrowing values across reviews (each review's text is idiosyncratic). NaNs are retained here and handled per-model in the modelling pipeline - native NaN support for tree-based models; 0-placeholder + missing indicator for models that cannot accept NaN.

**Scope of This Notebook**

- Common Feature Selection 
- Review imputation strategy for Set A (this will be revalidated on the training split during modelling pipeline)
- Construct missing-value indicators for Set A, Set C, and Set E2
- Select final columns per set
- Save each set as a separate CSV

**Common Features for All Sets**
- `Verified`, `Type Of Traveller`, `Seat Type`, `review_length`

## **1. Data Load**

In [153]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

df = pd.read_csv('../1_data/processed/05_absa_bert_scores.csv')
print(f"Loaded: {df.shape}")
df.info()

FileNotFoundError: [Errno 2] No such file or directory: '../1_data/processed/05_absa_bert_scores.csv'

## **2. Common Feature Selection**

**Statistical relevance and Deployment validity (available at review submission time) are applied as a criteria.**

As such, `Airline Name` and `Covid_Period` are intentionally excluded from the common feature set. `Airline Name` has high cardinality and would not generalize to airlines unseen during training; more importantly, including it risks acting as 
a shortcut that the model could use in place of aspect-level sentiment. `Covid_Period` is a categorical variable tied to a 
specific historical event and would not generalize to future reviews as the pandemic has ended.

In [ ]:
df_check = df.copy()

print("=" * 70)
print("1) Verified: association with target value (Recommended)")
print("=" * 70)

overall_gap = pd.crosstab(df_check['Verified'], df_check['Recommended'], normalize='index')[1]
gap = abs(overall_gap[True] - overall_gap[False])
print(f"Recommendation rate — Verified: {overall_gap[True]:.3f}, Non-verified: {overall_gap[False]:.3f}")
print(f"Gap: {gap:.3f} → {'meaningful association' if gap > 0.03 else 'weak/no association'}")

print("\n--- Controlling for Type Of Traveller ---")
sub = pd.crosstab([df_check['Type Of Traveller'], df_check['Verified']],
                   df_check['Recommended'], normalize='index')[1].unstack()
sub['gap'] = (sub[True] - sub[False]).round(3)
print(sub.round(3))
consistent_direction = (sub['gap'] < 0).sum()
print(f"→ Verified shows LOWER recommendation rate in {consistent_direction}/{len(sub)} traveller groups")

print("\n--- Controlling for Seat Type ---")
sub2 = pd.crosstab([df_check['Seat Type'], df_check['Verified']],
                    df_check['Recommended'], normalize='index')[1].unstack()
sub2['gap'] = (sub2[True] - sub2[False]).round(3)
print(sub2.round(3))
consistent_direction2 = (sub2['gap'] < 0).sum()
print(f"→ Verified shows LOWER recommendation rate in {consistent_direction2}/{len(sub2)} seat type groups")


print("\n" + "=" * 70)
print("2) Seat Type / Type Of Traveller: association with target (Recommended)")
print("=" * 70)

for group_col in ['Seat Type', 'Type Of Traveller']:
    rec_rate = df_check.groupby(group_col)['Recommended'].mean().round(3)
    spread = rec_rate.max() - rec_rate.min()
    print(f"\n--- {group_col} — Recommendation rate by group ---")
    print(rec_rate)
    print(f"→ Spread = {spread:.3f} → {'meaningful association' if spread > 0.05 else 'weak association'}")


print("\n" + "=" * 70)
print("3) review_length: sanity check")
print("=" * 70)
print(df_check['review_length'].describe().round(2))
print("→ No missing values, no invalid values.")

print("\n" + "=" * 70)
print("SUMMARY: Common Feature Candidates")
print("=" * 70)
summary = pd.DataFrame({
    'Feature': ['Verified', 'Type Of Traveller', 'Seat Type', 'review_length'],
    'Associated with target?': [
        'Yes' if gap > 0.03 else 'Weak',
        'Yes (see spread)', 'Yes (see spread)', 'N/A'
    ],
    'Deployment-safe?': ['Yes', 'Yes', 'Yes', 'Yes (derived from text)']
})
print(summary.to_string(index=False))

1) Verified: association with target value (Recommended)
Recommendation rate — Verified: 0.303, Non-verified: 0.370
Gap: 0.067 → meaningful association

--- Controlling for Type Of Traveller ---
Verified           False   True    gap
Type Of Traveller                     
Business           0.303  0.319  0.016
Couple Leisure     0.306  0.267 -0.039
Family Leisure     0.292  0.249 -0.043
Solo Leisure       0.394  0.349 -0.045
Unknown            0.448  0.000 -0.448
→ Verified shows LOWER recommendation rate in 4/5 traveller groups

--- Controlling for Seat Type ---
Verified         False   True    gap
Seat Type                           
Business Class   0.570  0.525 -0.045
Economy Class    0.392  0.273 -0.120
First Class      0.533  0.609  0.076
Premium Economy  0.359  0.331 -0.028
Unknown          0.029    NaN    NaN
→ Verified shows LOWER recommendation rate in 3/5 seat type groups

2) Seat Type / Type Of Traveller: association with target (Recommended)

--- Seat Type — Recommendation

## **3. Set A Imputation Strategy**

In [154]:
# 3.1 Missing Value Check

rating_cols = [
    'Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
    'Ground Service', 'Inflight Entertainment', 'Wifi & Connectivity',
    'Value For Money'
]

print(df[rating_cols].isnull().sum().to_frame('missing_count').assign(
    missing_pct=lambda x: (x['missing_count'] / len(df) * 100).round(1)
))

                        missing_count  missing_pct
Seat Comfort                     4217         18.4
Cabin Staff Service              4307         18.7
Food & Beverages                 8818         38.4
Ground Service                   4688         20.4
Inflight Entertainment          12735         55.4
Wifi & Connectivity             17084         74.3
Value For Money                  1174          5.1


> ### **Drop 'Inflight Entertainment' and 'Wifi & Connectivity'**
> `Inflight Entertainment` and `Wifi & Connectivity` will be excluded due to high missingness. `Food & Beverages` has fairly large missing values, but this needs to be investigated further to confirm whether imputation is acceptable or not.

In [155]:
# 3.2 Missing & Distribution Check

set_a_rating_cols = [
    'Seat Comfort',
    'Cabin Staff Service',
    'Food & Beverages',
    'Ground Service',
    'Value For Money'
]

print("=== Missing & Distribution Summary ===")
for col in set_a_rating_cols:
    missing_pct = df[col].isna().mean() * 100
    print(f"  {col:<25} missing: {missing_pct:.1f}%  median: {df[col].median()}  mean: {df[col].mean():.3f}")

=== Missing & Distribution Summary ===
  Seat Comfort              missing: 18.4%  median: 3.0  mean: 2.635
  Cabin Staff Service       missing: 18.7%  median: 3.0  mean: 2.886
  Food & Beverages          missing: 38.4%  median: 2.0  mean: 2.595
  Ground Service            missing: 20.4%  median: 1.0  mean: 2.349
  Value For Money           missing: 5.1%  median: 2.0  mean: 2.459


In [156]:
# 3.3 Overall vs Group Median & Mean Comparison

group_cols = ['Type Of Traveller', 'Seat Type']

for group in group_cols:
    print(f"\n=== Median by {group} (with Overall) ===")
    group_median = df.groupby(group)[set_a_rating_cols].median().round(2)
    overall_median = df[set_a_rating_cols].median().round(2)
    overall_median.name = 'Overall'
    combined_median = pd.concat([group_median, overall_median.to_frame().T])
    print(combined_median)

    print(f"\n=== Mean by {group} (with Overall) ===")
    group_mean = df.groupby(group)[set_a_rating_cols].mean().round(2)
    overall_mean = df[set_a_rating_cols].mean().round(2)
    overall_mean.name = 'Overall'
    combined_mean = pd.concat([group_mean, overall_mean.to_frame().T])
    print(combined_mean)


=== Median by Type Of Traveller (with Overall) ===
                Seat Comfort  Cabin Staff Service  Food & Beverages  \
Business                 3.0                  3.0               2.0   
Couple Leisure           2.0                  3.0               2.0   
Family Leisure           2.0                  2.0               2.0   
Solo Leisure             3.0                  3.0               3.0   
Unknown                  3.5                  4.0               3.0   
Overall                  3.0                  3.0               2.0   

                Ground Service  Value For Money  
Business                   2.0              2.0  
Couple Leisure             1.0              1.0  
Family Leisure             1.0              1.0  
Solo Leisure               2.0              2.0  
Unknown                    1.0              4.0  
Overall                    1.0              2.0  

=== Mean by Type Of Traveller (with Overall) ===
                Seat Comfort  Cabin Staff Service 

> ### **Key Insights**
>
> - **Mean-Median Gap (Skewness)**
>
>   Across most groups, the mean is lower than the median, indicating a right-skewed distribution; most respondents give low scores (1–2), while a minority give high scores (4–5), pulling the mean upward. This shows that **median imputation is a safer choice** than mean imputation for these columns.
>
> - **"Unknown" in Traveller Type, Seat Type**
>
>   Other categories show consistent directionality across all five rating dimensions (e.g., Economy Class scores lowest on every single dimension, while Business/First Class score highest on every dimension.) This consistency is what makes their group medians meaningful representations of a coherent group's experience.
>
>   However, the 'Unknown' category breaks this pattern. It scores at the top across four dimensions (Seat Comfort, Cabin Staff Service, Food & Beverages, Value For Money) but drops to the bottom on Ground Service alone. This directional inconsistency suggests 'Unknown' aggregates a mix of heterogeneous respondents rather than representing a coherent group, **so its own median is not a reliable estimate for imputing missing values within that group. As such, the overall median is used as a safer fallback for this category specifically.**

In [157]:
# 3.4 Before vs After Imputation: Overall vs Group-wise Median

df_check = df.copy()

group_cols = ['Type Of Traveller', 'Seat Type']


def print_imputation_table(title, imputed_series_dict):
    print(f"\n=== Before vs After Imputation ({title}) ===\n")
    print(f"{'Column':<25} {'Missing%':>9} {'Mean(orig)':>11} {'Mean(imp)':>10} {'MeanDiff':>9} {'Std(orig)':>10} {'Std(imp)':>9}")
    print("-" * 100)

    for col in set_a_rating_cols:
        original = df_check[col].dropna()
        missing_pct = df_check[col].isna().mean() * 100
        imputed = imputed_series_dict[col]

        print(f"{col:<25} {missing_pct:>8.1f}% "
              f"{original.mean():>11.3f} {imputed.mean():>10.3f} "
              f"{abs(original.mean()-imputed.mean()):>9.3f} "
              f"{original.std():>10.3f} {imputed.std():>9.3f}")


def group_median_with_unknown_fallback(df, col, group_col, unknown_label='Unknown'):
    overall_median = df[col].median()
    group_median = df.groupby(group_col)[col].transform('median')

    is_unknown = df[group_col] == unknown_label
    group_median_adjusted = group_median.where(~is_unknown, overall_median)

    return df[col].fillna(group_median_adjusted).fillna(overall_median)


# 3.4.1 Overall median imputation
overall_imputed = {col: df_check[col].fillna(df_check[col].median()) for col in set_a_rating_cols}
print_imputation_table("Overall", overall_imputed)

# 3.4.2 Group-wise median imputation (Unknown forced to use overall median)
for group in group_cols:
    group_imputed = {
        col: group_median_with_unknown_fallback(df_check, col, group)
        for col in set_a_rating_cols
    }
    print_imputation_table(f"Group: {group}", group_imputed)


=== Before vs After Imputation (Overall) ===

Column                     Missing%  Mean(orig)  Mean(imp)  MeanDiff  Std(orig)  Std(imp)
----------------------------------------------------------------------------------------------------
Seat Comfort                  18.4%       2.635      2.702     0.067      1.451     1.319
Cabin Staff Service           18.7%       2.886      2.908     0.021      1.592     1.435
Food & Beverages              38.4%       2.595      2.366     0.228      1.500     1.213
Ground Service                20.4%       2.349      2.074     0.275      1.594     1.522
Value For Money                5.1%       2.459      2.435     0.023      1.586     1.548

=== Before vs After Imputation (Group: Type Of Traveller) ===

Column                     Missing%  Mean(orig)  Mean(imp)  MeanDiff  Std(orig)  Std(imp)
----------------------------------------------------------------------------------------------------
Seat Comfort                  18.4%       2.635      2.66

> ### **Key Insights**
>
> Although imputation itself uses the median, **`MeanDiff` (the shift in the dataset's overall mean before vs. after imputation) is used as the primary evaluation criterion. (i.e. lower = better preservation of the original distribution)**, since comparing median values directly offers little discriminative power.
>
> By this criterion, **Type Of Traveller outperforms Seat Type** across all columns by a notable margin.
>
> Yet, this is worth double-checking: `Unknown` is forced to use the overall median in both groupings, so if missing values are disproportionately concentrated in `Unknown` for Type Of Traveller relative to Seat Type, its apparent advantage could reflect a large share of its imputed values simply mirroring Overall imputation.
>
> **As such, to isolate the real grouping effect, the comparison is re-run excluding `Unknown` from both variables (known-only comparison).**

In [158]:
# 3.5 Known-only MeanDiff: pure group explanatory power (Unknown excluded)

def known_only_meandiff(df, group_col, cols, unknown_label='Unknown'):
    df_known = df[df[group_col] != unknown_label].copy()
    results = {}
    for col in cols:
        original = df_known[col].dropna()
        group_median = df_known.groupby(group_col)[col].transform('median')
        imputed = df_known[col].fillna(group_median)
        results[col] = round(abs(original.mean() - imputed.mean()), 3)
    return results

seat_type_known = known_only_meandiff(df, 'Seat Type', set_a_rating_cols)
traveller_known = known_only_meandiff(df, 'Type Of Traveller', set_a_rating_cols)

comparison_df = pd.DataFrame({
    'Seat Type (known-only)': seat_type_known,
    'Traveller Type (known-only)': traveller_known
})
print(comparison_df)

                     Seat Type (known-only)  Traveller Type (known-only)
Seat Comfort                          0.072                        0.012
Cabin Staff Service                   0.026                        0.011
Food & Beverages                      0.184                        0.061
Ground Service                        0.188                        0.049
Value For Money                       0.002                        0.000


> ### **Final Imputation Strategy for Set A**
>
> With `Unknown` excluded entirely from both variables, **Type Of Traveller still outperforms Seat Type across all five columns**.
>
> This rules out the earlier concern that Type Of Traveller's advantage was an artifact of `Unknown` being disproportionately mapped to the overall median. Since the gap persists once `Unknown` is removed from the comparison, **Type Of Traveller is confirmed as the stronger grouping variable for imputation**, despite Seat Type showing a visibly wider raw spread in the median/mean tables (Section 3.3). 
>
> This counterintuitive result is treated as a genuine, empirically-grounded finding rather than a data artifact, though **it is re-validated on the training split during pipeline construction before finalizing the imputation method per column.**

In [159]:
# 3.6 Ensure the original dataset was not changed

na_original = df[set_a_rating_cols].isna().sum()
na_check = df_check[set_a_rating_cols].isna().sum()

comparison = pd.DataFrame({
    'df (original)': na_original,
    'df_check (copy)': na_check,
    'match': na_original == na_check
})
print(comparison)

assert na_original.equals(na_check)
print("\ndf unchanged — verification passed.")

                     df (original)  df_check (copy)  match
Seat Comfort                  4217             4217   True
Cabin Staff Service           4307             4307   True
Food & Beverages              8818             8818   True
Ground Service                4688             4688   True
Value For Money               1174             1174   True

df unchanged — verification passed.


## **4. Construct Missing-Value Indicators**

In [ ]:
def add_missing_indicators(df, cols, suffix='_missing'):
    for col in cols:
        df[f'{col}{suffix}'] = df[col].isna().astype(int)
    return df

# Set A
set_a_rating_cols = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages', 'Ground Service', 'Value For Money']
df = add_missing_indicators(df, set_a_rating_cols)

# Set C - rule-based aspect VADER
set_c_cols = ['aspect_seat', 'aspect_food', 'aspect_staff', 'aspect_ground_service', 'aspect_entertainment']  
df = add_missing_indicators(df, set_c_cols)

# Set E2 — keyword-gated ABSA-BERT
set_e2_cols = ['absa_seat_e2', 'absa_food_e2', 'absa_staff_e2', 'absa_ground_service_e2', 'absa_entertainment_e2'] 
df = add_missing_indicators(df, set_e2_cols)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22980 entries, 0 to 22979
Data columns (total 51 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Airline Name                    22980 non-null  object 
 1   Verified                        22980 non-null  bool   
 2   Type Of Traveller               22980 non-null  object 
 3   Seat Type                       22980 non-null  object 
 4   Seat Comfort                    18763 non-null  float64
 5   Cabin Staff Service             18673 non-null  float64
 6   Food & Beverages                14162 non-null  float64
 7   Ground Service                  18292 non-null  float64
 8   Inflight Entertainment          10245 non-null  float64
 9   Wifi & Connectivity             5896 non-null   float64
 10  Value For Money                 21806 non-null  float64
 11  Recommended                     22980 non-null  int64  
 12  Covid_Period                    

## **5. Set Features**

In [ ]:
common_cols = ['Verified', 'Type Of Traveller', 'Seat Type', 'review_length']
target_col = ['Recommended']

# Set A: Numerical sub-ratings
set_a_core = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
              'Ground Service', 'Value For Money']
set_a_indicators = [f'{c}_missing' for c in set_a_core]
set_a_df = df[set_a_core + set_a_indicators + common_cols + target_col].copy()

# Set B: VADER document-level scores
set_b_core = ['vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']
set_b_df = df[set_b_core + common_cols + target_col].copy()

# Set C: Rule-based aspect VADER scores
set_c_core = ['aspect_seat', 'aspect_food', 'aspect_staff',
              'aspect_ground_service', 'aspect_entertainment']
set_c_indicators = [f'{c}_missing' for c in set_c_core]
set_c_df = df[set_c_core + set_c_indicators + common_cols + target_col].copy()

# Set D: B + C
set_d_core = set_b_core + set_c_core
set_d_df = df[set_d_core + set_c_indicators + common_cols + target_col].copy()

# Set E1: ABSA-BERT full inference scores
set_e1_core = ['absa_seat', 'absa_food', 'absa_staff',
               'absa_ground_service', 'absa_entertainment']
set_e1_df = df[set_e1_core + common_cols + target_col].copy()

# Set E2: ABSA-BERT keyword-gated scores
set_e2_core = ['absa_seat_e2', 'absa_food_e2', 'absa_staff_e2',
               'absa_ground_service_e2', 'absa_entertainment_e2']
set_e2_indicators = [f'{c}_missing' for c in set_e2_core]
set_e2_df = df[set_e2_core + set_e2_indicators + common_cols + target_col].copy()

# Save all sets
import os

output_dir = 'data/final_sets/'
os.makedirs(output_dir, exist_ok=True)

sets_to_save = {
    '01_set_a.csv': set_a_df,
    '02_set_b.csv': set_b_df,
    '03_set_c.csv': set_c_df,
    '04_set_d.csv': set_d_df,
    '05_set_e1.csv': set_e1_df,
    '06_set_e2.csv': set_e2_df,
}

for filename, set_df in sets_to_save.items():
    set_df.to_csv(f'{output_dir}{filename}', index=False)
    print(f"{filename:<12} shape={set_df.shape}")

01_set_a.csv shape=(22980, 15)
02_set_b.csv shape=(22980, 9)
03_set_c.csv shape=(22980, 15)
04_set_d.csv shape=(22980, 19)
05_set_e1.csv shape=(22980, 10)
06_set_e2.csv shape=(22980, 15)
